In [2]:
import pandas as pd

x_train = pd.read_csv('x_train_final.csv')
y_train = pd.read_csv('y_train_final.csv')
x_test  = pd.read_csv('x_test_final.csv')

print(f"\nx_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}")


x_train: (667264, 12)
y_train: (667264, 2)
x_test: (20657, 11)


In [3]:
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)

# premières lignes
print("\n--- x_train ---")
print(x_train.head())

print("\n--- y_train ---")
print(y_train.head())

print("\n--- x_test ---")
print(x_test.head())

x_train: (667264, 12)
y_train: (667264, 2)
x_test: (20657, 11)

--- x_train ---
   Unnamed: 0.1  Unnamed: 0   train gare        date  arret  p2q0  p3q0  p4q0  \
0             0           0  VBXNMF  KYF  2023-04-03      8   0.0   0.0   1.0   
1             1           1  VBXNMF  JLR  2023-04-03      9   0.0   0.0   0.0   
2             2           2  VBXNMF  EOH  2023-04-03     10  -1.0   0.0   0.0   
3             3           3  VBXNMF  VXY  2023-04-03     11  -1.0  -1.0   0.0   
4             4           4  VBXNMF  OCB  2023-04-03     12  -1.0  -1.0  -1.0   

   p0q2  p0q3  p0q4  
0  -3.0  -1.0  -2.0  
1   1.0   0.0   1.0  
2  -1.0   0.0   0.0  
3   2.0  -2.0   0.0  
4  -1.0   3.0   2.0  

--- y_train ---
   Unnamed: 0  p0q0
0           0  -1.0
1           1  -1.0
2           2  -1.0
3           3   1.0
4           4   3.0

--- x_test ---
   Unnamed: 0   train gare        date  arret  p2q0  p3q0  p4q0  p0q2  p0q3  \
0           0  ZPQEKP  VXY  2023-11-13     12   0.0   0.0  -2.0  -4.0

In [4]:
# infos colonnes
print("\n--- infos x_train ---")
print(x_train.info())

# statistiques
print("\n--- stats x_train ---")
print(x_train.describe())


--- infos x_train ---
<class 'pandas.DataFrame'>
RangeIndex: 667264 entries, 0 to 667263
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0.1  667264 non-null  int64  
 1   Unnamed: 0    667264 non-null  int64  
 2   train         667264 non-null  str    
 3   gare          667264 non-null  str    
 4   date          667264 non-null  str    
 5   arret         667264 non-null  int64  
 6   p2q0          667264 non-null  float64
 7   p3q0          667264 non-null  float64
 8   p4q0          667264 non-null  float64
 9   p0q2          667264 non-null  float64
 10  p0q3          667264 non-null  float64
 11  p0q4          667264 non-null  float64
dtypes: float64(6), int64(3), str(3)
memory usage: 73.2 MB
None

--- stats x_train ---
        Unnamed: 0.1     Unnamed: 0          arret           p2q0  \
count  667264.000000  667264.000000  667264.000000  667264.000000   
mean   333631.500000  333631.500000 

In [5]:
print(y_train.value_counts())

Unnamed: 0  p0q0
0           -1.0    1
1           -1.0    1
2           -1.0    1
3            1.0    1
4            3.0    1
                   ..
667259       1.0    1
667260       2.0    1
667261       2.0    1
667262       1.0    1
667263       1.0    1
Name: count, Length: 667264, dtype: int64


In [6]:
print(y_train.describe())

          Unnamed: 0           p0q0
count  667264.000000  667264.000000
mean   333631.500000      -0.159950
std    192622.669348       1.987872
min         0.000000    -160.000000
25%    166815.750000      -1.000000
50%    333631.500000       0.000000
75%    500447.250000       1.000000
max    667263.000000      15.000000


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# load
x_train = pd.read_csv("x_train_final.csv")
y_train = pd.read_csv("y_train_final.csv")
x_test  = pd.read_csv("x_test_final.csv")

y = y_train["p0q0"]

# concat pour encoder pareil
full = pd.concat([x_train, x_test], axis=0)

# drop index inutiles
full = full.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")

# date features
full["date"] = pd.to_datetime(full["date"])
full["jour"] = full["date"].dt.day
full["mois"] = full["date"].dt.month
full["jour_semaine"] = full["date"].dt.dayofweek
full = full.drop(columns=["date"])

# label encoding
for col in ["train","gare"]:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col])

# re-split
x_train = full.iloc[:len(y)]
x_test  = full.iloc[len(y):]

# model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

model.fit(x_train, y)

pred_test = model.predict(x_test)

In [8]:
from sklearn.metrics import mean_absolute_error

# prédictions sur le train
pred_train = model.predict(x_train)

# MAE
mae_train = mean_absolute_error(y, pred_train)
print("MAE train:", mae_train)

# prédictions sur le train
pred_train = model.predict(x_train)

# MAE
mae_train = mean_absolute_error(y, pred_train)
print("MAE train:", mae_train)

MAE train: 0.2919786621187416
MAE train: 0.2919786621187416


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x_tr, x_val, y_tr, y_val = train_test_split(x_train, y, test_size=0.2, random_state=42)

model.fit(x_tr, y_tr)
pred_val = model.predict(x_val)

mae_val = mean_absolute_error(y_val, pred_val)
print("MAE validation:", mae_val)

MAE validation: 0.7805413891032797


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

x_tr, x_val, y_tr, y_val = train_test_split(x_train, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,  # limite overfitting
    n_jobs=-1,
    random_state=42
)

model.fit(x_tr, y_tr)
pred_val = model.predict(x_val)
print("MAE validation:", mean_absolute_error(y_val, pred_val))

MAE validation: 0.8001807502218841
